In [1]:
import os 
import glob

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from datasets import load_dataset 
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from natsort import natsorted

load_dotenv()
HF_USERNAME = os.getenv("HF_USERNAME")
DATA_ROOT = os.getenv("DATA_ROOT")

/data/dtce-schmidt/phys2526/venvs/wwdc_spectra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cols = ['z', 'zErr', 'rChi2', 'velDisp', 'velDispErr', 'extinction_r', 'petroMag_r', 'psfMag_r', 'psfMagErr_r', 'modelMag_u', 'modelMagErr_u', 'modelMag_g', 'modelMagErr_g', 'modelMag_r', 'modelMagErr_r', 'modelMag_i', 'modelMagErr_i', 'modelMag_z', 'modelMagErr_z', 'petroR50_r', 'petroR90_r', 'nii_6584_flux', 'nii_6584_flux_err', 'h_alpha_flux', 'h_alpha_flux_err', 'oiii_5007_flux', 'oiii_5007_flux_err', 'h_beta_flux', 'h_beta_flux_err', 'h_delta_flux', 'h_delta_flux_err', 'd4000', 'd4000_err', 'bptclass', 'lgm_tot_p50', 'sfr_tot_p50']

# Load data

In [3]:
from astroML.datasets import fetch_sdss_specgals
data = fetch_sdss_specgals()
print(f"Loaded {data.shape[0]} galaxies.")

Loaded 661598 galaxies.


In [4]:
fp = f"{DATA_ROOT}/sdss_II/spender_I_flow_v2/embeddings/7655991_0"
train_files = natsorted(glob.glob(f"{fp}/train/*.parquet"))
test_files = natsorted(glob.glob(f"{fp}/test/*.parquet"))
val_files = natsorted(glob.glob(f"{fp}/val/*.parquet"))

# 2. Define the file path mapping using the explicitly sorted lists
data_files = {
    "train": train_files,
    "test": test_files,
    "val": val_files
}
# 3. Load the files into a single DatasetDict
ds = load_dataset("parquet", data_files=data_files)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['orig', 'cond', 'uncond', 'id', 'BESTOBJID', 'z', 'ra', 'dec', 'mask_ratio'],
        num_rows: 422481
    })
    test: Dataset({
        features: ['orig', 'cond', 'uncond', 'id', 'BESTOBJID', 'z', 'ra', 'dec', 'mask_ratio'],
        num_rows: 52811
    })
    val: Dataset({
        features: ['orig', 'cond', 'uncond', 'id', 'BESTOBJID', 'z', 'ra', 'dec', 'mask_ratio'],
        num_rows: 52810
    })
})


In [5]:
df_sdss = pd.DataFrame(data)
df_sdss['merge_id'] = df_sdss['specObjID'].astype('int64')

merged_dfs = {}
aligned_arrays = {}

splits = ['train', 'test', 'val']
for split in splits:
    print(f"--- Processing '{split}' split ---")
    # Convert the current Hugging Face split to a Pandas DataFrame
    df_spender = ds[split].to_pandas()
    
    # Clean the Hugging Face IDs (strip the 'b', quotes, and spaces)
    df_spender['merge_id'] = df_spender['id'].astype(str).str.extract(r'(\d+)')[0].astype('int64')
    
    # Perform the master merge
    matched_df = pd.merge(df_spender, df_sdss, on='merge_id', how='inner')
    
    # Clean up by dropping the temporary merge column
    matched_df = matched_df.drop(columns=['merge_id'])
    
    initial_len = len(matched_df)
    
    # Apply the mask ratio filter (drop any rows where mask_ratio is 1.0)
    if "mask_ratio" in matched_df.columns:
        matched_df = matched_df[matched_df["mask_ratio"] <= 0.5]
        
    # Reset index for clean dataframes
    matched_df = matched_df.reset_index(drop=True)
    
    # Store the full pandas dataframe in our dictionary
    merged_dfs[split] = matched_df
    
    dropped_count = initial_len - len(matched_df)
    print(f"Successfully merged {initial_len} galaxies.")
    if dropped_count > 0:
        print(f"Dropped {dropped_count} rows with mask_ratio == 1.0. Final count: {len(matched_df)}")

print("All splits successfully matched, masked, and aligned!")

--- Processing 'train' split ---
Successfully merged 367609 galaxies.
Dropped 13726 rows with mask_ratio == 1.0. Final count: 353883
--- Processing 'test' split ---
Successfully merged 45990 galaxies.
Dropped 1723 rows with mask_ratio == 1.0. Final count: 44267
--- Processing 'val' split ---
Successfully merged 46028 galaxies.
Dropped 1660 rows with mask_ratio == 1.0. Final count: 44368
All splits successfully matched, masked, and aligned!


# Linear  Regression

In [6]:
# Maybe log fluxes and add to the dataframe. Means the linear function logic is not as complex.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

def fit_lin_fn(
    feature,
    embed_type
):
    # Check if the feature is a flux early on so we can use it in data cleaning
    is_flux = "flux" in feature.lower()

    # 1. Clean the data: Drop missing values
    train_clean = merged_dfs["train"].dropna(subset=[feature])
    test_clean = merged_dfs["test"].dropna(subset=[feature])

    # --- NEW: Remove standard SDSS missing data flags (-9999.0) ---
    train_clean = train_clean[train_clean[feature] != -9999.0]
    test_clean = test_clean[test_clean[feature] != -9999.0]

    # --- UPDATED: Only enforce strictly positive values if we are doing a log10 transform ---
    if is_flux:
        train_clean = train_clean[train_clean[feature] > 0]
        test_clean = test_clean[test_clean[feature] > 0]

    print(f"\n--- Running: {embed_type} ---")
    
    # 2. Extract arrays
    if embed_type == "cond+z":
        X_train = np.stack(train_clean["cond"].values)
        y_train = train_clean[feature].values

        X_test = np.stack(test_clean["cond"].values)
        y_test = test_clean[feature].values

        z_train = train_clean["z_x"].values.reshape(-1, 1)
        z_test = test_clean["z_x"].values.reshape(-1, 1)
        
        z_scaler = StandardScaler()
        z_train_scaled = z_scaler.fit_transform(z_train)
        z_test_scaled = z_scaler.transform(z_test)

        if len(X_train.shape) == 3:
            X_train = X_train.reshape(X_train.shape[0], -1)
            X_test = X_test.reshape(X_test.shape[0], -1)

        X_train = np.hstack((X_train, z_train_scaled))
        X_test = np.hstack((X_test, z_test_scaled))
        
    else:
        X_train = np.stack(train_clean[embed_type].values)
        y_train = train_clean[feature].values

        X_test = np.stack(test_clean[embed_type].values)
        y_test = test_clean[feature].values

        if len(X_train.shape) == 3:
            X_train = X_train.reshape(X_train.shape[0], -1)
            X_test = X_test.reshape(X_test.shape[0], -1)

    # 3. Conditionally Log10 AND Standard Scale the Target Variable
    y_scaler = None

    if is_flux:
        y_train = np.log10(y_train)
        y_test = np.log10(y_test)
        
        #y_scaler = #StandardScaler()
        y_train = y_train.reshape(-1, 1).flatten() #y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_test = y_test.reshape(-1, 1).flatten() #y_scaler.transform(y_test.reshape(-1, 1)).flatten()
    else:
        #y_scaler = StandardScaler()
        #y_train = y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        #y_test = y_scaler.transform(y_test.reshape(-1, 1)).flatten()
        y_train = y_train.reshape(-1, 1).flatten() #y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_test = y_test.reshape(-1, 1).flatten()
    
    #X_scaler = StandardScaler()
    X_train_scaled = X_train #X_scaler.fit_transform(X_train) 
    X_test_scaled = X_test # X_scaler.transform(X_test) 

    # 5. Train the Linear Regression Model
    model = LinearRegression() 
    print(f"Training on {X_train_scaled.shape[0]} samples..., {X_train_scaled.shape[1]} features")
    model.fit(X_train_scaled, y_train)

    # Extract coefficients and intercept
    model_coefs = model.coef_.tolist()
    model_intercept = float(model.intercept_)

    # 6. Make predictions 
    predictions = model.predict(X_test_scaled)

    # 8. Bootstrapping for Errors
    n_iterations = 1000
    bootstrapped_r2 = []
    bootstrapped_mse = []

    for _ in range(n_iterations):
        y_test_resampled, preds_resampled = resample(y_test, predictions)
        bootstrapped_r2.append(r2_score(y_test_resampled, preds_resampled))
        bootstrapped_mse.append(mean_squared_error(y_test_resampled, preds_resampled))

    # Calculate statistics 
    r2_mean = np.mean(bootstrapped_r2)
    r2_median = np.median(bootstrapped_r2)
    r2_std = np.std(bootstrapped_r2)
    r2_ci_lower = np.percentile(bootstrapped_r2, 2.5)
    r2_ci_upper = np.percentile(bootstrapped_r2, 97.5)

    mse_mean = np.mean(bootstrapped_mse)
    mse_median = np.median(bootstrapped_mse)
    mse_std = np.std(bootstrapped_mse)
    mse_ci_lower = np.percentile(bootstrapped_mse, 2.5)
    mse_ci_upper = np.percentile(bootstrapped_mse, 97.5)

    # Return the results with Coefficients and Intercept
    return {
        "Feature": feature,
        "Embed_Type": embed_type,
        "R2_Mean": r2_mean,
        "R2_Median": r2_median,
        "R2_Std": r2_std,
        "R2_95%_CI_Lower": r2_ci_lower,
        "R2_95%_CI_Upper": r2_ci_upper,
        "MSE_Mean": mse_mean,
        "MSE_Median": mse_median,
        "MSE_Std": mse_std,
        "MSE_95%_CI_Lower": mse_ci_lower,
        "MSE_95%_CI_Upper": mse_ci_upper,
        "Intercept": model_intercept,
        "Coefficients": model_coefs 
    }

In [8]:
# Define the parameters you want to iterate over
features_to_test = ["z_x", "h_alpha_flux", "velDisp", "d4000", "extinction_r", 'petroR50_r', 'petroR90_r', 'nii_6584_flux', 'oiii_5007_flux', 'h_beta_flux', 'h_delta_flux', 'lgm_tot_p50', 'sfr_tot_p50']
embed_types = ["orig", "cond", "uncond", "cond+z"]

# Create an empty list to store the dictionaries
all_results = []

print("Starting model runs...")

# Iterate over features and embedding types
for feature in features_to_test:
    for embed_type in embed_types:
        print(f"Processing Feature: {feature} | Embed: {embed_type}")
        
        # Run the function and capture the output dictionary
        result_dict = fit_lin_fn(feature=feature, embed_type=embed_type)
        
        # Add it to our master list
        all_results.append(result_dict)

# Convert the list of dictionaries into a Pandas DataFrame
results_df = pd.DataFrame(all_results)

# Display the table
print("\n--- Final Results Table ---")
print(results_df)

# Optional: Save the table to a CSV file for your records
results_df.to_csv("model_bootstrap_results_no_scaling.csv", index=False)
print("\nResults successfully saved to 'model_bootstrap_results_no_scaling.csv'")

Starting model runs...
Processing Feature: z_x | Embed: orig

--- Running: orig ---
Training on 353883 samples..., 10 features
Processing Feature: z_x | Embed: cond

--- Running: cond ---
Training on 353883 samples..., 10 features
Processing Feature: z_x | Embed: uncond

--- Running: uncond ---
Training on 353883 samples..., 10 features
Processing Feature: z_x | Embed: cond+z

--- Running: cond+z ---
Training on 353883 samples..., 11 features
Processing Feature: h_alpha_flux | Embed: orig

--- Running: orig ---
Training on 342506 samples..., 10 features
Processing Feature: h_alpha_flux | Embed: cond

--- Running: cond ---
Training on 342506 samples..., 10 features
Processing Feature: h_alpha_flux | Embed: uncond

--- Running: uncond ---
Training on 342506 samples..., 10 features
Processing Feature: h_alpha_flux | Embed: cond+z

--- Running: cond+z ---
Training on 342506 samples..., 11 features
Processing Feature: velDisp | Embed: orig

--- Running: orig ---
Training on 353883 samples..

In [10]:
# 1. Function to calculate asymmetric errors and format as LaTeX
def format_latex_r2(row):
    mean = row['R2_Mean']
    ci_lower = row['R2_95%_CI_Lower']
    ci_upper = row['R2_95%_CI_Upper']
    
    # Calculate relative distances for the error bars
    err_up = ci_upper - mean
    err_low = mean - ci_lower
    
    # Format string: 0.850^{+0.012}_{-0.015}
    return f"${mean:.3f}^{{+{err_up:.3f}}}_{{-{err_low:.3f}}}$"

# Apply formatting to create a new column
results_df['R2_LaTeX'] = results_df.apply(format_latex_r2, axis=1)

# --- NEW: 2. Split the DataFrame ---
# Create a boolean mask for features that contain "flux"
is_flux = results_df['Feature'].str.contains('flux', case=False, na=False)

# Split into two dataframes
df_flux = results_df[is_flux]
df_non_flux = results_df[~is_flux]

# --- NEW: 3. Helper function to build a LaTeX table from a DataFrame ---
def build_latex_table(df, caption, label):
    latex_str = "\\begin{table}[htbp]\n"
    latex_str += "\\centering\n"
    latex_str += f"\\caption{{{caption}}}\n"
    latex_str += f"\\label{{{label}}}\n"
    latex_str += "\\begin{tabular}{llc}\n"
    latex_str += "\\toprule\n"
    latex_str += "\\textbf{Feature} & \\textbf{Embedding} & \\textbf{$R^2$} \\\\\n"
    latex_str += "\\midrule\n"

    for feature, group in df.groupby("Feature", sort=False):
        # Escape underscores for LaTeX
        feature_name = feature.replace("_", "\\_")
        
        first_row = True
        for _, row in group.iterrows():
            embed = row['Embed_Type']
            r2_val = row['R2_LaTeX']
            
            if first_row:
                latex_str += f"\\multirow{{{len(group)}}}{{*}}{{{feature_name}}} & {embed} & {r2_val} \\\\\n"
                first_row = False
            else:
                latex_str += f" & {embed} & {r2_val} \\\\\n"
        
        latex_str += "\\midrule\n"

    # Clean up trailing midrule and close
    latex_str = latex_str.rsplit("\\midrule\n", 1)[0]
    latex_str += "\\bottomrule\n"
    latex_str += "\\end{tabular}\n"
    latex_str += "\\end{table}\n"
    
    return latex_str

# 4. Generate the strings
table_flux = build_latex_table(
    df=df_flux, 
    caption="Model Performance for Flux Features", 
    label="tab:flux_results"
)

table_non_flux = build_latex_table(
    df=df_non_flux, 
    caption="Model Performance for Non-Flux Features", 
    label="tab:non_flux_results"
)

# 5. Print and Save
print("--- FLUX TABLE ---")
print(table_flux)

print("\n--- NON-FLUX TABLE ---")
print(table_non_flux)

# Optional: Save both to a single text file
with open("split_r2_tables.tex", "w") as f:
    f.write("% --- FLUX TABLE ---\n")
    f.write(table_flux)
    f.write("\n\n% --- NON-FLUX TABLE ---\n")
    f.write(table_non_flux)

--- FLUX TABLE ---
\begin{table}[htbp]
\centering
\caption{Model Performance for Flux Features}
\label{tab:flux_results}
\begin{tabular}{llc}
\toprule
\textbf{Feature} & \textbf{Embedding} & \textbf{$R^2$} \\
\midrule
\multirow{4}{*}{h\_alpha\_flux} & orig & $0.734^{+0.008}_{-0.009}$ \\
 & cond & $0.641^{+0.008}_{-0.008}$ \\
 & uncond & $0.715^{+0.008}_{-0.010}$ \\
 & cond+z & $0.753^{+0.009}_{-0.011}$ \\
\midrule
\multirow{4}{*}{nii\_6584\_flux} & orig & $0.628^{+0.007}_{-0.008}$ \\
 & cond & $0.625^{+0.008}_{-0.008}$ \\
 & uncond & $0.659^{+0.008}_{-0.008}$ \\
 & cond+z & $0.680^{+0.007}_{-0.009}$ \\
\midrule
\multirow{4}{*}{oiii\_5007\_flux} & orig & $0.515^{+0.008}_{-0.009}$ \\
 & cond & $0.390^{+0.008}_{-0.007}$ \\
 & uncond & $0.426^{+0.008}_{-0.008}$ \\
 & cond+z & $0.509^{+0.008}_{-0.008}$ \\
\midrule
\multirow{4}{*}{h\_beta\_flux} & orig & $0.663^{+0.007}_{-0.007}$ \\
 & cond & $0.516^{+0.007}_{-0.007}$ \\
 & uncond & $0.595^{+0.007}_{-0.007}$ \\
 & cond+z & $0.640^{+0.007}_{-